# Principal component analysis: geometry, computation, and supervised pipelines

**Lecture 16 · Notebook 01 · CMOR 438 / INDE 577**

The 8×8 handwritten digits in the course database have 64 pixel
features. This laboratory asks:

> **Can a lower-dimensional linear representation preserve useful digit
> structure while reducing the cost of a supervised model?**

We derive PCA from variance and reconstruction viewpoints, implement it
with the singular value decomposition, visualize principal directions,
and evaluate a leakage-safe PCA + $k$-nearest-neighbor pipeline.

## How to use this notebook

Use the **Rice DSM** kernel and run cells in order. Predict the geometry
before viewing each plot. The live core route takes about 100 minutes;
the practice and engineering extensions take about 40 minutes.

**Route:** inspect images → center → derive projection → compute SVD →
explained variance → reconstruction → train-only pipeline → accuracy and
latency tradeoff → professional release boundary.

## Learning objectives

You will be able to:

1. derive PCA's maximum-variance and minimum-reconstruction objectives;
2. connect covariance eigenvectors to the SVD of a centered data matrix;
3. distinguish centering, standardization, projection, and whitening;
4. interpret scores, loadings, explained variance, and reconstruction;
5. place PCA inside a supervised pipeline without leakage; and
6. compare predictive quality, dimensionality, fit cost, and inference cost.

## Why this matters

High-dimensional data appear in imaging, spectroscopy, genomics,
telemetry, and text. Redundant coordinates can make storage, distance
calculations, visualization, and downstream fitting expensive or
unstable. PCA provides a transparent linear baseline.

It is not a universal denoiser. The highest-variance directions may be
irrelevant to a supervised target, rare signals may live in low-variance
directions, and linear projections cannot unfold every manifold.

## Professional practice

Fit every data-dependent transformation using training observations
only. Version the fitted mean, components, scaling policy, source schema,
and retained-rank rule with the supervised model. Measure end-to-end
behavior: PCA may reduce prediction cost while increasing training cost.

### Where PCA appears in real systems

- **Quality inspection:** compress correlated sensor channels before anomaly
  review, while monitoring reconstruction residuals.
- **Genomics and spectroscopy:** expose dominant modes of variation and detect
  batch effects before fitting scientific models.
- **Image retrieval:** shorten vectors used by a distance-based search service.
- **Model engineering:** reduce memory or prediction work when a downstream
  estimator scales with the number of coordinates.

These are different products. The first may value reconstruction error, the
second interpretability, and the last latency at acceptable predictive quality.
PCA has no single “best” rank until the operational objective is named.

In [ ]:
# Import local data, plotting tools, and timing support.
import sqlite3
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import course_database_path

RANDOM_STATE = 438
DATABASE_PATH = course_database_path()
assert DATABASE_PATH.is_file(), "Run the course refresh command first."
sns.set_theme(style="whitegrid", context="notebook")

## 1. Worked example: images represented as vectors

Each database row contains 64 grayscale intensities and a digit label.
The pixels can be reshaped to an $8\times8$ image for people, but models
receive a vector in $\mathbb{R}^{64}$. PCA will learn from the pixels;
labels are reserved for visualization and downstream supervised tests.

### Notation used throughout

| Symbol | Shape | Meaning |
|---|---:|---|
| $X$ | $n\times p$ | observations by features |
| $\bar{x}$ | $p$ | training-feature means |
| $X_c$ | $n\times p$ | centered training matrix |
| $V_k$ | $p\times k$ | first $k$ orthonormal loading directions |
| $Z=X_cV_k$ | $n\times k$ | component scores for observations |
| $\widehat X_c=ZV_k^\top$ | $n\times p$ | rank-$k$ centered reconstruction |

Rows are observations throughout this course. Some references use observations
as columns; their formulas look transposed but express the same geometry.

In [ ]:
# Query the real scikit-learn digits observations stored in course SQLite.
with sqlite3.connect(DATABASE_PATH) as connection:
    digits = pd.read_sql_query(
        "SELECT * FROM digits_observations ORDER BY observation_id",
        connection,
    )

pixel_columns = [f"pixel_{index}" for index in range(64)]
digit_pixels = digits[pixel_columns].to_numpy(dtype=float)
digit_labels = digits["digit_label"].to_numpy(dtype=int)

print("Observations × pixels:", digit_pixels.shape)
print("Pixel range:", digit_pixels.min(), "to", digit_pixels.max())
print("Class counts:", np.bincount(digit_labels))
assert digit_pixels.shape == (1797, 64)
assert set(np.unique(digit_labels)) == set(range(10))

In [ ]:
# Restore the spatial arrangement only for human inspection.
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5), constrained_layout=True)
for digit, axis in enumerate(axes.flat):
    example = digit_pixels[np.flatnonzero(digit_labels == digit)[0]]
    axis.imshow(example.reshape(8, 8), cmap="magma", vmin=0, vmax=16)
    axis.set_title(f"digit {digit}")
    axis.axis("off")
fig.suptitle("Real handwritten observations: 64 intensities per image")
plt.show()

## 2. PCA begins by centering

Let $X\in\mathbb{R}^{n\times p}$ contain observations as rows. Compute the
training mean $\bar{x}\in\mathbb{R}^p$ and centered matrix

$$
X_c=X-\mathbf{1}\bar{x}^{\top}.
$$

For a unit direction $v\in\mathbb{R}^p$, observation $i$ receives the
scalar score $z_i=x_{c,i}^{\top}v$. All scores together are $z=X_cv$.
Because centered scores have mean zero, their sample variance is

$$
\begin{aligned}
\operatorname{Var}(X_cv)
&=\frac{1}{n-1}(X_cv)^{\top}(X_cv)\\
&=\frac{1}{n-1}v^{\top}X_c^{\top}X_cv\\
&=v^{\top}Sv,
\end{aligned}
\qquad
S=\frac{1}{n-1}X_c^{\top}X_c.
$$

Thus PCA's first direction solves the constrained optimization problem

$$
\max_{v\in\mathbb{R}^p} v^{\top}Sv
\quad\text{subject to}\quad v^{\top}v=1.
$$

The constraint prevents the meaningless solution of multiplying $v$ by an
arbitrarily large number. With Lagrangian

$$
\mathcal{L}(v,\lambda)=v^{\top}Sv-\lambda(v^{\top}v-1),
$$

stationarity gives

$$
\nabla_v\mathcal{L}=2Sv-2\lambda v=0
\quad\Longrightarrow\quad Sv=\lambda v.
$$

For an eigenvector $v$, the objective equals $v^{\top}Sv=\lambda$.
Therefore the largest eigenvalue supplies the maximum-variance direction.
Later directions solve the same problem while remaining orthogonal to earlier
directions.

In [ ]:
# Use a two-feature example to make projection and residuals visible.
geometry = np.array([
    [-2.4, -1.5], [-1.8, -1.2], [-1.1, -0.3], [-0.4, -0.5],
    [0.2, 0.3], [0.8, 0.4], [1.2, 1.1], [1.9, 1.2], [2.4, 1.8],
])
centered_geometry = geometry - geometry.mean(axis=0)
_, _, geometry_vt = np.linalg.svd(centered_geometry, full_matrices=False)
first_direction = geometry_vt[0]
scores = centered_geometry @ first_direction
reconstructions = np.outer(scores, first_direction)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(centered_geometry[:, 0], centered_geometry[:, 1], s=55,
           label="centered observations", zorder=3)
axis_points = np.outer(np.linspace(scores.min()-0.4, scores.max()+0.4, 2),
                       first_direction)
ax.plot(axis_points[:, 0], axis_points[:, 1], color="#c75b39", linewidth=2.5,
        label="first principal direction")
for point, reconstruction in zip(centered_geometry, reconstructions, strict=True):
    ax.plot([point[0], reconstruction[0]], [point[1], reconstruction[1]],
            color="#7d8790", alpha=0.75, linewidth=1)
ax.scatter(reconstructions[:, 0], reconstructions[:, 1], marker="x",
           color="#00205b", label="rank-one reconstructions", zorder=4)
ax.set(aspect="equal", xlabel="centered feature 1", ylabel="centered feature 2",
       title="PCA projection: scores along an axis, residuals perpendicular")
ax.legend()
plt.show()

assert np.isclose(np.linalg.norm(first_direction), 1.0)
assert np.allclose(reconstructions @ first_direction, scores)

### Read the geometry before moving on

The blue points are centered observations. Each gray segment is a residual
from an observation to its orthogonal projection on the orange axis. The score
is the signed coordinate **along** that axis; the reconstruction maps that
coordinate back into feature space. PCA selects the axis whose total squared
residual length is smallest—equivalently, whose scores have the largest
variance.

## 3. The reconstruction and SVD views

Collect $k$ orthonormal loading vectors in
$V_k\in\mathbb{R}^{p\times k}$, so $V_k^{\top}V_k=I_k$. The matrix
$P_k=V_kV_k^{\top}$ is an orthogonal projector:

$$
P_k^{\top}=P_k,
\qquad
P_k^2=P_k.
$$

Scores and centered reconstructions are

$$
Z=X_cV_k,
\qquad
\widehat X_c=ZV_k^{\top}=X_cP_k.
$$

PCA minimizes the total squared reconstruction residual,

$$
\min_{V_k^{\top}V_k=I_k}
\left\lVert X_c-X_cV_kV_k^{\top}\right\rVert_F^2.
$$

Now write the singular value decomposition as

$$
X_c=U\Sigma V^{\top},
$$

where columns of $U$ are left singular vectors, columns of $V$ are right
singular vectors, and singular values satisfy
$\sigma_1\geq\sigma_2\geq\cdots\geq0$. The
Eckart--Young--Mirsky theorem states that truncating this decomposition gives
the best rank-$k$ approximation in Frobenius norm:

$$
X_{c,k}=U_k\Sigma_kV_k^{\top}.
$$

Therefore PCA loadings are the leading columns of $V$, scores are
$U_k\Sigma_k=X_cV_k$, and explained variances satisfy

$$
\lambda_j=\frac{\sigma_j^2}{n-1}.
$$

The covariance-eigenvector, maximum-variance, minimum-reconstruction, and SVD
descriptions are four views of the same linear subspace.

In [ ]:
from rice_dsm.ml.decomposition import PrincipalComponentProjector

# Verify the course implementation against the equations above.
teaching_pca = PrincipalComponentProjector(n_components=2)
geometry_scores = teaching_pca.fit_transform(geometry)
geometry_reconstruction = teaching_pca.inverse_transform(geometry_scores)

print("Components:\n", teaching_pca.components_)
print("Explained-variance ratios:", teaching_pca.explained_variance_ratio_)
print("Rank-two reconstruction error:",
      teaching_pca.reconstruction_error(geometry))
assert np.allclose(geometry_reconstruction, geometry)
assert np.allclose(teaching_pca.components_ @ teaching_pca.components_.T,
                   np.eye(2))

### Centering is required; scaling is a policy

PCA on covariance is sensitive to measurement scale. Standardization
makes each feature's sample scale comparable before PCA, but can amplify
low-variance noise and discard meaningful physical units. Images here
share a common intensity scale, so we center without standardizing for
the visual analysis. In the supervised benchmark we make the pipeline's
scaling choice explicit.

**Whitening** goes further by dividing component scores by their standard
deviations. It gives retained coordinates unit variance but removes the
variance magnitudes and can magnify weak directions. PCA itself does not
imply whitening.

## 4. Fit on training data before choosing the representation

The test set represents future observations. Its means, variances, or
directions must not influence the fitted representation. We split first,
then fit PCA and any scaler inside a pipeline using training data only.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratification preserves digit proportions without revealing test pixels.
pixels_train, pixels_test, labels_train, labels_test = train_test_split(
    digit_pixels, digit_labels, test_size=0.30, stratify=digit_labels,
    random_state=RANDOM_STATE,
)
assert pixels_train.shape[0] + pixels_test.shape[0] == len(digit_pixels)
assert set(np.unique(labels_train)) == set(range(10))

In [ ]:
from sklearn.decomposition import PCA

# Fit the visual PCA on training images only; labels are not passed to fit.
digit_pca = PCA(n_components=0.95, svd_solver="full")
train_scores = digit_pca.fit_transform(pixels_train)
test_scores = digit_pca.transform(pixels_test)
cumulative_variance = np.cumsum(digit_pca.explained_variance_ratio_)

print(f"Retained components: {digit_pca.n_components_} of 64")
print(f"Training variance retained: {cumulative_variance[-1]:.2%}")
assert train_scores.shape[1] == digit_pca.n_components_
assert test_scores.shape[1] == digit_pca.n_components_
assert cumulative_variance[-1] >= 0.95

In [ ]:
# A scree plot shows marginal and cumulative variance, not target usefulness.
component_numbers = np.arange(1, digit_pca.n_components_ + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].bar(component_numbers, digit_pca.explained_variance_ratio_,
            color="#5b8db8")
axes[0].set(title="Scree plot", xlabel="principal component",
            ylabel="fraction of training variance")
axes[1].plot(component_numbers, cumulative_variance, marker="o", markersize=3)
axes[1].axhline(0.95, color="#c75b39", linestyle="--", label="95% rule")
axes[1].set(title="Cumulative explained variance", xlabel="components retained",
            ylabel="cumulative fraction", ylim=(0, 1.02))
axes[1].legend()
plt.show()

### How to read these curves

The left panel shows the marginal variance associated with each direction.
The right panel accumulates those fractions. The familiar “elbow” and 95%
rules are descriptive heuristics, not laws. They do not know whether a faint
stroke distinguishes a 3 from an 8, whether latency matters, or whether a
low-variance scientific effect is the quantity of interest.

In [ ]:
# Loadings can be reshaped into eigen-images that describe directions,
# not literal or average handwritten digits.
fig, axes = plt.subplots(2, 5, figsize=(10, 4.8), constrained_layout=True)
for index, axis in enumerate(axes.flat):
    loading_image = digit_pca.components_[index].reshape(8, 8)
    limit = np.abs(loading_image).max()
    axis.imshow(loading_image, cmap="coolwarm", vmin=-limit, vmax=limit)
    axis.set_title(f"PC {index + 1}")
    axis.axis("off")
fig.suptitle("Leading PCA loading directions ('eigendigits')")
plt.show()

In [ ]:
# Use the first two scores as a labeled audit view, never as proof of clusters.
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(train_scores[:, 0], train_scores[:, 1], c=labels_train,
                     cmap="tab10", s=18, alpha=0.72)
ax.set(title="Two-component view of training digits",
       xlabel="principal component 1", ylabel="principal component 2")
fig.colorbar(scatter, ax=ax, ticks=range(10), label="digit audit label")
plt.show()

two_component_fraction = cumulative_variance[1]
print(f"Only {two_component_fraction:.1%} of training variance is visible here.")
assert 0.0 < two_component_fraction < cumulative_variance[-1]

In [ ]:
# Reconstruction makes the information tradeoff concrete.
reconstruction_ranks = [2, 10, 20, digit_pca.n_components_]
example_images = pixels_test[:6]
fig, axes = plt.subplots(len(reconstruction_ranks) + 1, len(example_images),
                         figsize=(10, 8), constrained_layout=True)
for column, image in enumerate(example_images):
    axes[0, column].imshow(image.reshape(8, 8), cmap="magma", vmin=0, vmax=16)
    axes[0, column].axis("off")
axes[0, 0].set_ylabel("original", rotation=0, labelpad=35, va="center")

reconstruction_errors = []
for row, rank in enumerate(reconstruction_ranks, start=1):
    approximation = (
        (example_images - digit_pca.mean_) @ digit_pca.components_[:rank].T
    ) @ digit_pca.components_[:rank] + digit_pca.mean_
    reconstruction_errors.append(np.mean((example_images - approximation) ** 2))
    for column, image in enumerate(approximation):
        axes[row, column].imshow(image.reshape(8, 8), cmap="magma", vmin=0, vmax=16)
        axes[row, column].axis("off")
    axes[row, 0].set_ylabel(f"rank {rank}", rotation=0, labelpad=35, va="center")
fig.suptitle("More components reduce reconstruction error")
plt.show()

assert np.all(np.diff(reconstruction_errors) < 0)

In [ ]:
# Quantify reconstruction on the complete held-out set. Every rank reuses the
# PCA directions fitted only on training images.
candidate_ranks = sorted(
    rank
    for rank in {2, 5, 10, 20, 30, digit_pca.n_components_}
    if rank <= digit_pca.n_components_
)
held_out_mse = []
for rank in candidate_ranks:
    centered_test = pixels_test - digit_pca.mean_
    low_rank_scores = centered_test @ digit_pca.components_[:rank].T
    reconstructed_test = low_rank_scores @ digit_pca.components_[:rank]
    reconstructed_test += digit_pca.mean_
    held_out_mse.append(np.mean((pixels_test - reconstructed_test) ** 2))

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(candidate_ranks, held_out_mse, marker="o", color="#00205b")
ax.set(
    title="Held-out reconstruction error falls as rank increases",
    xlabel="retained principal components",
    ylabel="mean squared pixel reconstruction error",
)
plt.show()

assert np.all(np.diff(held_out_mse) < 0)

Reconstruction error must decrease as nested PCA ranks grow, which makes it a
useful implementation invariant. But lower reconstruction error does not imply
higher classification accuracy: reconstruction weights every squared pixel
residual, whereas classification values distinctions associated with the
target.

## 5. PCA inside a supervised pipeline

We compare $k$-nearest-neighbor classification in 64 standardized pixel
coordinates with classification after PCA. The pipeline guarantees that
the scaler and PCA fit only on training observations and that prediction
applies the exact same transformations.

This is a particularly meaningful pairing: nearest-neighbor prediction
repeatedly computes distances to training observations, so fewer
coordinates can reduce inference work. We will measure rather than assume
an accuracy or speed improvement.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Define two auditable candidates with the same split and neighbor rule. The
# only structural difference is whether PCA compresses the scaled pixels before
# KNN computes distances.
full_knn = Pipeline(
    [
        ("scale", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5)),
    ]
)
pca_knn = Pipeline(
    [
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.90, svd_solver="full")),
        ("classifier", KNeighborsClassifier(n_neighbors=5)),
    ]
)

# fit() estimates every training-dependent quantity. predict() applies those
# frozen transformations to observations that were not used during fitting.
full_knn.fit(pixels_train, labels_train)
pca_knn.fit(pixels_train, labels_train)
full_predictions = full_knn.predict(pixels_test)
pca_predictions = pca_knn.predict(pixels_test)

full_accuracy = accuracy_score(labels_test, full_predictions)
pca_accuracy = accuracy_score(labels_test, pca_predictions)
retained = pca_knn.named_steps["pca"].n_components_

print(f"Full 64-pixel accuracy: {full_accuracy:.3f}")
print(f"PCA {retained}-component accuracy: {pca_accuracy:.3f}")
assert full_accuracy > 0.90
assert pca_accuracy > 0.90

In [ ]:
# Accuracy can hide which classes changed. Row-normalized confusion matrices
# estimate P(predicted class | true class) for each digit in the test set.
full_confusion = confusion_matrix(labels_test, full_predictions, normalize="true")
pca_confusion = confusion_matrix(labels_test, pca_predictions, normalize="true")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
sns.heatmap(full_confusion, vmin=0, vmax=1, cmap="Blues", ax=axes[0], cbar=False)
sns.heatmap(pca_confusion, vmin=0, vmax=1, cmap="Blues", ax=axes[1], cbar=False)
sns.heatmap(
    pca_confusion - full_confusion,
    center=0,
    cmap="coolwarm",
    ax=axes[2],
    cbar_kws={"label": "PCA minus full"},
)
axes[0].set_title("Scaled KNN")
axes[1].set_title("Scaled PCA + KNN")
axes[2].set_title("Change by true/predicted class")
for axis in axes:
    axis.set(xlabel="predicted digit", ylabel="true digit")
plt.show()

full_recall = np.diag(full_confusion)
pca_recall = np.diag(pca_confusion)
class_evidence = pd.DataFrame(
    {
        "digit": np.arange(10),
        "full_recall": full_recall,
        "pca_recall": pca_recall,
        "change": pca_recall - full_recall,
    }
)
display(class_evidence.round(3))

### Why class-level evidence matters

Two systems can have nearly identical overall accuracy while failing on
different populations. Here each class is a digit; in an industrial system it
might be a failure type, customer cohort, scanner, laboratory, or geographic
region. A release decision should state which slices are safety- or
business-critical before inspecting results, then report uncertainty where the
sample size is small.

In [ ]:
def median_prediction_seconds(model, observations, repeats=9):
    """Measure warmed batch prediction time with a robust median.

    Timing a single cold call confounds model work with one-time imports,
    allocation, and cache effects. Repetition does not make this a universal
    benchmark, but it makes the local comparison less fragile.
    """
    model.predict(observations[:10])  # Warm the prediction path.

    durations = []
    for _ in range(repeats):
        start = perf_counter()
        model.predict(observations)
        durations.append(perf_counter() - start)

    return float(np.median(durations))


# Time the complete prediction pipelines, including scaling and PCA transform.
full_seconds = median_prediction_seconds(full_knn, pixels_test)
pca_seconds = median_prediction_seconds(pca_knn, pixels_test)
speed_ratio = full_seconds / pca_seconds

benchmark = pd.DataFrame(
    [
        {
            "candidate": "scaled KNN",
            "features_at_KNN": 64,
            "accuracy": full_accuracy,
            "batch_predict_ms": 1000 * full_seconds,
        },
        {
            "candidate": "scaled PCA + KNN",
            "features_at_KNN": retained,
            "accuracy": pca_accuracy,
            "batch_predict_ms": 1000 * pca_seconds,
        },
    ]
)
display(benchmark.round(3))
print(f"Observed prediction speed ratio: {speed_ratio:.1f}×")
assert benchmark["batch_predict_ms"].gt(0).all()

Timings vary with hardware, operating system, load, and library versions;
the ratio observed in this run is evidence about this environment, not a
universal constant. PCA has a fit-time cost, stores its mean and loading
matrix, and may reduce accuracy. Its value depends on the number of
training examples, predictions, retained components, and operational
constraints.

The test labels may evaluate the two candidates once. They must not be
reused to tune the variance threshold, neighbor count, or scaling policy.
Those choices belong inside cross-validation on the training set.

## 6. The supervised application changes the selection question

Explained variance is unsupervised: it does not know which directions
predict the digit. A 95% variance threshold optimizes neither accuracy
nor latency. In a professional study, candidate ranks are selected within
cross-validation and the untouched test set estimates the complete
selection procedure.

A useful evaluation report includes predictive metric and uncertainty,
retained dimension, fit and prediction timing, artifact size,
reconstruction evidence if scientifically relevant, and slice-level
errors. A performance gain can mean comparable accuracy at materially
lower latency; it need not mean a larger accuracy number.

### A release decision has at least two axes

Suppose a digit-retrieval service has a latency objective and a maximum
acceptable accuracy loss. The team should define those thresholds before the
final test comparison. It can then choose among Pareto-efficient candidates:
no retained candidate should be both slower and less accurate than another.

The fitted pipeline—not a loose list of preprocessing instructions—is the
deployable unit. Its artifact must preserve feature order, scaler statistics,
PCA mean and components, neighbor configuration, label map, training-data
provenance, and compatible library versions. Production monitoring should
separate service latency, schema failures, score/reconstruction drift, class
mix, and delayed outcome quality.

## Guided practice: a dimension–quality curve

Using training-only cross-validation, compare 10, 20, 30, and 40 PCA
components for KNN. Plot validation accuracy against retained dimension.
Success means the scaler and PCA are inside each fold's pipeline and you
identify a Pareto-efficient candidate rather than choosing on the test set.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Each fold independently fits scaling and PCA, preventing validation leakage.
folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
dimension_rows = []
for components in [10, 20, 30, 40]:
    candidate = Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=components, svd_solver="full")),
        ("classifier", KNeighborsClassifier(n_neighbors=5)),
    ])
    scores = cross_val_score(candidate, pixels_train, labels_train,
                             cv=folds, scoring="accuracy")
    dimension_rows.append({"components": components,
                           "mean_accuracy": scores.mean(),
                           "fold_sd": scores.std(ddof=1)})
dimension_evidence = pd.DataFrame(dimension_rows)
display(dimension_evidence)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.errorbar(dimension_evidence["components"],
            dimension_evidence["mean_accuracy"],
            yerr=dimension_evidence["fold_sd"], marker="o", capsize=4)
ax.set(title="Training-only dimension selection",
       xlabel="PCA components", ylabel="cross-validated accuracy")
plt.show()
assert dimension_evidence["mean_accuracy"].between(0, 1).all()

## Independent practice

Compare centering-only PCA with standardized PCA. Explain why constant
corner pixels create zero-variance coordinates and why that does not make
standardization automatically wrong or right. Success requires separate
claims about numerical behavior, image semantics, and supervised evidence.

## Extension: student contribution boundary

Extend `PrincipalComponentProjector` in your companion project with one
reviewed capability: a variance-threshold rank rule, randomized SVD, or
an incremental fit for batches too large for memory. Add analytic tests
for orthonormal components, centered scores, monotone reconstruction
error, invalid shapes, and use before fit. Differentially compare the
learned subspace—not component signs alone—with a trusted implementation.
Keep the applied scientific story in a notebook that imports the package.

## Debugging and common failure modes

1. **Different component signs:** $v$ and $-v$ represent the same axis;
   compare subspaces or sign-align before comparing pictures.
2. **PCA accuracy is disappointing:** variance and target information are
   different objectives; inspect rank selection and task suitability.
3. **Validation looks implausibly strong:** verify split-first pipelines.
4. **One physical variable controls components:** inspect units and defend
   the centering/scaling policy.
5. **Timing changes across runs:** warm the code path, repeat measurements,
   report hardware/software context, and avoid tiny single timings.
6. **Sparse data become huge:** centered dense PCA may be inappropriate;
   consider sparse-compatible truncated SVD and its different semantics.

## Retrieval practice

1. Derive $\operatorname{Var}(X_cv)=v^{\top}Sv$ and state every matrix shape.
2. Why does the unit-norm constraint matter in the variance objective?
3. How do PCA scores and loadings differ?
4. What theorem justifies truncated-SVD reconstruction?
5. Why can two valid PCA implementations return opposite component signs?
6. Why must PCA be inside cross-validation?
7. Give one setting where lower variance may contain important signal.
8. Why inspect class-level performance when overall accuracy barely changes?
9. What evidence supports calling PCA a performance improvement here?
10. Which fitted objects must travel with a PCA-based production model?

## Takeaway

PCA is an optimal *linear* low-rank representation under a stated squared
reconstruction geometry. It can expose structure and reduce downstream cost,
but explained variance is not predictive value. Treat centering, scaling,
rank selection, evaluation, and artifact versioning as explicit engineering
decisions.

## Further reading

- [scikit-learn PCA documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [scikit-learn pipeline documentation](https://scikit-learn.org/stable/modules/compose.html#pipeline)
- [NumPy SVD documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)
- [scikit-learn model evaluation guide](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Python `perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)